<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/XGBoostt_HPV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    balanced_accuracy_score
)

from imblearn.over_sampling import SMOTE

In [41]:
SEED = 44

In [42]:
df = pd.read_excel("HPV2025.xlsx")

print(df.shape)

df.head()

(726, 17)


,PatientID,CenterID,Task 1,Task 2,Task 3,Age,Gender,Tobacco Consumption,Alcohol Consumption,Performance Status,Treatment,T-stage,N-stage,M-stage,HPV Status,Relapse,RFS
0,CHUM-001,1,1,1,0,82.0,1,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1704.0
1,CHUM-002,1,1,1,0,73.0,1,NaN,NaN,NaN,1.0,T3,N2,M0,NaN,1.0,439.0
2,CHUM-006,1,1,1,0,65.0,1,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1186.0
3,CHUM-007,1,1,1,0,70.0,0,NaN,NaN,NaN,0.0,T2,N2,M0,NaN,0.0,1702.0
4,CHUM-008,1,1,1,0,67.0,0,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1499.0


In [43]:
df = df.dropna(subset=["HPV Status"])

In [44]:
tobacco_mode = df["Tobacco Consumption"].mode()[0]

df["Tobacco Consumption"] = df[
    "Tobacco Consumption"
].fillna(tobacco_mode)

In [45]:
alcohol_mode = df["Alcohol Consumption"].mode()[0]

df["Alcohol Consumption"] = df[
    "Alcohol Consumption"
].fillna(alcohol_mode)

In [46]:
df = df.drop(
    columns=[
        "Performance Status",
        "Relapse",
        "RFS",
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

print(df.shape)

(562, 9)


In [47]:
df["T-stage"] = df["T-stage"].replace({
    "T0":0,
    "T1":1,
    "T2":2,
    "T3":3,
    "T4":4
})

df["N-stage"] = df["N-stage"].replace({
    "N0":0,
    "N1":1,
    "N2":2,
    "N3":3
})

df["M-stage"] = df["M-stage"].replace({
    "M0":0,
    "M1":1
})

/tmp/ipykernel_1653/2205384798.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"] = df["T-stage"].replace({
/tmp/ipykernel_1653/2205384798.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"] = df["N-stage"].replace({
/tmp/ipykernel_1653/2205384798.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_sile

In [48]:
X = df[
    [
        "Age",
        "Gender",
        "Tobacco Consumption",
        "Alcohol Consumption",
        "Treatment",
        "T-stage",
        "N-stage",
        "M-stage"
    ]
]

y = df["HPV Status"]

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED
)

In [50]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [51]:
smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(pd.Series(y_train_smote).value_counts())

HPV Status
1.0    403
0.0    403
Name: count, dtype: int64


In [52]:
!pip install xgboost

In [53]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=SEED,
    eval_metric="logloss"
)

model.fit(
    X_train_smote,
    y_train_smote
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [54]:
predicted = model.predict(X_test)

probabilities = model.predict_proba(X_test)[:,1]

In [55]:
print(
    classification_report(
        y_test,
        predicted
    )
)

              precision    recall  f1-score   support

         0.0       0.21      0.30      0.25        10
         1.0       0.93      0.89      0.91       103

    accuracy                           0.84       113
   macro avg       0.57      0.60      0.58       113
weighted avg       0.87      0.84      0.85       113



In [56]:
cm = confusion_matrix(
    y_test,
    predicted
)

print(cm)

[[ 3  7]
 [11 92]]


In [57]:
auc = roc_auc_score(
    y_test,
    probabilities
)

print("AUC =", auc)

AUC = 0.7106796116504854


In [58]:
bal_acc = balanced_accuracy_score(
    y_test,
    predicted
)

print("Balanced Accuracy =", bal_acc)

Balanced Accuracy = 0.5966019417475729
